# AI Interior Design Platform - Colab Pipeline & Training

This notebook contains two sections:
1. **Inference Pipeline**: Run the redesign system on existing models.
2. **Training Suite**: Fine-tune YOLOv8 and Stable Diffusion (LoRA) on custom interior datasets.

## 🛠 Setup & Dependencies

In [ ]:
!pip install -q ultralytics diffusers transformers accelerate torch torchvision opencv-python pillow xformers kaggle

### 🔑 Kaggle API Setup
1. Go to your Kaggle Account and click on 'Create New API Token'.
2. This will download a `kaggle.json` file.
3. Run the cell below to upload that file.

In [ ]:
from google.colab import files
import os

if not os.path.exists('/root/.kaggle/kaggle.json'):
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

## 🧠 Section 1: Inference Pipeline
Run this to see the system in action using pretrained weights.

In [ ]:
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from PIL import Image
import numpy as np
import cv2
from ultralytics import YOLO

# Setup Models
device = "cuda" if torch.cuda.is_available() else "cpu"
detection_model = YOLO('yolov8n.pt')

controlnet = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16 if device == "cuda" else torch.float32)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", controlnet=controlnet, torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to(device)

def get_canny_image(image):
    image_np = np.array(image)
    image_canny = cv2.Canny(image_np, 100, 200)
    image_canny = image_canny[:, :, None]
    image_canny = np.concatenate([image_canny, image_canny, image_canny], axis=2)
    return Image.fromarray(image_canny)

def redesign_room(image_path, style="modern"):
    image = Image.open(image_path).convert("RGB")
    
    # Detection
    results = detection_model(image)
    labels = [detection_model.names[int(box.cls)] for box in results[0].boxes]
    objects_str = ", ".join(set(labels))
    
    # Prompting
    prompt = f"A professional photo of a {style} room with {objects_str}. 8k, realistic."
    
    # Generation
    canny = get_canny_image(image)
    result = pipe(prompt, image=canny, num_inference_steps=25).images[0]
    return result

## 🏗 Section 2: Training Suite (The Model Part)
Use this section to train your own models for Detection and Generation.

### 1. Data Collection
Download public datasets for the four styles: Scandinavian, Boho, Modern, Industrial.

In [ ]:
# Option 1: Downloading ADE20K Subset (Common for room understanding)
!wget -O ade20k_subset.zip https://path-to-public-interior-dataset.zip
!unzip -q ade20k_subset.zip -d ./data

# Option 2: Download Kaggle Interior Design Dataset
!kaggle datasets download -d arashnic/interior-design-dataset
!unzip -q interior-design-dataset.zip -d ./data

### 2. Fine-tune YOLOv8 (Detection)
Train the model to recognize specific furniture in your custom dataset.

In [ ]:
from ultralytics import YOLO

# Load a pretrained model
custom_yolo = YOLO('yolov8n.pt')

# Start training
custom_yolo.train(
    data='/content/data/custom_interior.yaml', # Path to your YOLO yaml
    epochs=50,
    imgsz=640,
    device=0
)

### 3. Fine-tune Stable Diffusion (Generation via LoRA)
Teach the model your specific aesthetics (Modern, Boho, etc.).

In [ ]:
# Download the training script from Diffusers
!wget https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth_lora.py

# Train for a specific style (e.g., Scandinavian)
!accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --instance_data_dir="/content/data/scandinavian_images" \
  --output_dir="/content/models/lora_scandi" \
  --instance_prompt="a photo of a room in scandinavian style" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --max_train_steps=500

### 4. Load & Use Custom Models
Combine your newly trained detection weights and LoRA styles.

In [ ]:
# Load custom detection weights
my_detection_model = YOLO('/content/runs/detect/interior_redesign/weights/best.pt')

# Load LoRA weights into the pipe
pipe.load_lora_weights("/content/models/lora_scandi/pytorch_lora_weights.safetensors")

# Run with custom training
result = redesign_room("my_living_room.jpg", style="scandinavian")
result.show()